# PRISM — End-to-end demo

Predictive Retrieval with Inductive Subgraph Memory.

Pipeline:
1. Generate synthetic KGQA data (mimics GNN-RAG output)
2. Stage 1 — distill LAS to match teacher distributions
3. Stage 2 — REINFORCE fine-tune with coverage/efficiency/diversity reward
4. Evaluate Hit@1, F1, ECE, Brier, mean #paths
5. Plot Pareto cost-quality curve

Runtime ~10 min on a T4 GPU with default config.

## 0. Setup

In [1]:
# If running in Colab: clone the repo first, e.g.
# !git clone <your_repo>.git
# %cd <your_repo>

!pip install -q torch numpy scikit-learn tqdm matplotlib pyyaml

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/ThS/Research/colab')

# Import với tên COLAB (uppercase)
from COLAB.graph_rag.prism.config import CFG
print('CFG loaded:', CFG.d_model)


CFG loaded: 256


In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from COLAB.graph_rag.prism.config import CFG
from COLAB.graph_rag.prism.data import SyntheticKGQA
from COLAB.graph_rag.prism.las import LAS
from COLAB.graph_rag.prism.controller import deterministic_select
from COLAB.graph_rag.prism.distill import train_distill
from COLAB.graph_rag.prism.rl import train_rl
from COLAB.graph_rag.prism.evaluate import evaluate, baseline_full_subgraph
from COLAB.graph_rag.prism.metrics import reliability_diagram

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## 1. Configure for fast demo run

Override defaults in `config.py` for a Colab-friendly run. Bump these for real experiments.

In [5]:
CFG.n_train = 800
CFG.n_eval = 200
CFG.n_relations = 32
CFG.max_paths = 30
CFG.n_candidates = 24
CFG.d_model = 192
CFG.n_layers = 3
CFG.distill_epochs = 3
CFG.rl_steps = 200
CFG.rl_n_samples = 3
print('config:', CFG)

config: Config(seed=42, device='cuda', data_mode='synthetic', data_path='', n_train=800, n_eval=200, n_relations=32, n_entities=4096, max_paths=30, max_path_len=3, n_candidates=24, d_model=192, d_query=384, n_heads=4, n_layers=3, dropout=0.1, k_options=[5, 10, 20, 50], max_select_paths=10, stopping_eps=0.05, diversity_lambda=0.3, distill_epochs=3, distill_lr=0.0003, distill_batch_size=16, distill_temp=1.0, rl_steps=200, rl_lr=5e-05, rl_n_samples=3, rl_clip_norm=0.5, rl_w_coverage=0.6, rl_w_efficiency=0.2, rl_w_diversity=0.2, rl_calib_weight=0.1, out_dir='outputs', ckpt_distill='prism_distill_best.pt', ckpt_rl='prism_rl_best.pt')


## 2. Generate synthetic KGQA data

In [6]:
train_set = SyntheticKGQA(CFG.n_train, cfg=CFG, seed=CFG.seed)
eval_set = SyntheticKGQA(CFG.n_eval, cfg=CFG, seed=CFG.seed + 100)
print(f'train={len(train_set)}  eval={len(eval_set)}')

s = train_set[0]
print('Sample 0:')
print('  query_emb shape:', s.query_emb.shape)
print('  paths shape:', s.paths.shape, '(N_paths × max_path_len)')
print('  candidates:', s.candidate_ids.shape[0])
print('  gold entities:', s.gold_entities.tolist())

train=800  eval=200
Sample 0:
  query_emb shape: torch.Size([384])
  paths shape: torch.Size([25, 3]) (N_paths × max_path_len)
  candidates: 24
  gold entities: [33, 1972, 3308]


## 3. Initialize LAS

The Lightweight Answer Simulator. Goal: mimic teacher LLM's `P(answer | query, retrieved_subgraph)`.

In [7]:
las = LAS(CFG).to(device)
n_params = sum(p.numel() for p in las.parameters() if p.requires_grad)
print(f'LAS trainable params: {n_params:,}')

LAS trainable params: 2,142,754


### 3a. Sanity check: forward pass before training

In [8]:
s = eval_set[0]
for k in ['query_emb', 'paths', 'path_to_candidate', 'candidate_gnn_scores',
         'candidate_ids', 'gold_entities']:
    setattr(s, k, getattr(s, k).to(device))

with torch.no_grad():
    probs, conf, _ = las(s.query_emb, s.paths, s.path_to_candidate, s.candidate_gnn_scores)
print('output prob shape:', probs.shape, 'sum:', float(probs.sum()))
print('confidence:', float(conf))
print('top-3 candidate ids:', s.candidate_ids[probs.topk(3).indices].tolist())
print('gold ids:', s.gold_entities.tolist())

output prob shape: torch.Size([24]) sum: 0.9999999403953552
confidence: 0.5582106113433838
top-3 candidate ids: [776, 3218, 2750]
gold ids: [33, 1972, 3308]


## 4. Stage 1 — distillation

In [9]:
ckpt_distill = train_distill(las, train_set, eval_set, cfg=CFG)
print('saved:', ckpt_distill)

distill ep0: 100%|██████████| 50/50 [00:35<00:00,  1.42it/s, loss=0.153]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[distill] epoch 0  train_loss=0.1534  eval_kl=0.1643
  saved best -> outputs/prism_distill_best.pt


distill ep1: 100%|██████████| 50/50 [00:34<00:00,  1.43it/s, loss=0.142]


[distill] epoch 1  train_loss=0.1418  eval_kl=0.1663


distill ep2: 100%|██████████| 50/50 [00:34<00:00,  1.45it/s, loss=0.139]


[distill] epoch 2  train_loss=0.1387  eval_kl=0.1666
saved: outputs/prism_distill_best.pt


### 4a. Evaluate after Stage 1

In [ ]:
metrics_after_distill = evaluate(las, eval_set, CFG)
for k, v in metrics_after_distill.items():
    print(f'  {k:14s} = {v}')

## 5. Stage 2 — REINFORCE fine-tune

Reward = 0.6·coverage + 0.2·efficiency + 0.2·diversity.

In [ ]:
ckpt_rl = train_rl(las, train_set, eval_set, cfg=CFG)
print('saved:', ckpt_rl)

## 6. Final evaluation

In [ ]:
prism_metrics = evaluate(las, eval_set, CFG)
base_metrics = baseline_full_subgraph(eval_set)

print('PRISM:')
for k, v in prism_metrics.items():
    print(f'  {k:14s} = {v}')
print()
print('Baseline (full subgraph):')
for k, v in base_metrics.items():
    print(f'  {k:14s} = {v}')
print()
print('Δ Hit@1   = {:+.4f}'.format(prism_metrics['hit@1'] - base_metrics['hit@1']))
cost_save = 1 - prism_metrics['mean_paths'] / base_metrics['mean_paths']
print(f'Cost save = {cost_save*100:.1f}% paths fewer')

## 7. Calibration analysis (reliability diagram)

In [ ]:
from COLAB.graph_rag.prism.metrics import expected_calibration_error, brier_score, hit_at_k

confs, corrs = [], []
with torch.no_grad():
    for sample in eval_set.samples:
        for k in ['query_emb', 'paths', 'path_to_candidate', 'candidate_gnn_scores',
                  'candidate_ids', 'gold_entities']:
            setattr(sample, k, getattr(sample, k).to(device))
        trace = deterministic_select(las, sample, CFG)
        confs.append(float(trace.confidence.item()))
        corrs.append(hit_at_k(trace.final_probs, sample.candidate_ids,
                              sample.gold_entities, k=1))

print(f'ECE = {expected_calibration_error(confs, corrs):.4f}')
print(f'Brier = {brier_score(confs, corrs):.4f}')

centers, accs, counts = reliability_diagram(confs, corrs, n_bins=10)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', label='perfectly calibrated', alpha=0.4)
ax.bar(centers, accs, width=0.08, alpha=0.6, edgecolor='black', label='LAS')
ax.set_xlabel('confidence')
ax.set_ylabel('actual accuracy')
ax.set_title('PRISM Reliability Diagram')
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()

## 8. Pareto cost-quality frontier (sweep stopping ε)

In [ ]:
eps_grid = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
frontier = []

for eps in eps_grid:
    CFG.stopping_eps = eps
    m = evaluate(las, eval_set, CFG)
    frontier.append((m['mean_paths'], m['hit@1']))
    print(f'eps={eps:>6.3f}  paths={m["mean_paths"]:5.2f}  hit@1={m["hit@1"]:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
xs, ys = zip(*frontier)
ax.plot(xs, ys, 'o-', label='PRISM Pareto')
ax.scatter([base_metrics['mean_paths']], [base_metrics['hit@1']],
           color='red', s=80, label='Baseline (full subgraph)', zorder=5)
ax.set_xlabel('Mean paths used (cost)')
ax.set_ylabel('Hit@1 (quality)')
ax.set_title('PRISM cost-quality Pareto frontier')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 9. Inspect a single query trace (qualitative)

In [ ]:
CFG.stopping_eps = 0.05
sample = eval_set[3]
for k in ['query_emb', 'paths', 'path_to_candidate', 'candidate_gnn_scores',
         'candidate_ids', 'gold_entities']:
    setattr(sample, k, getattr(sample, k).to(device))

trace = deterministic_select(las, sample, CFG)
print(f'Total available paths: {sample.paths.size(0)}')
print(f'Selected: {len(trace.selected_path_idx)} paths  K={trace.chosen_K}  reason={trace.stopped_reason}')
print(f'Confidence: {float(trace.confidence):.3f}')
print(f'KL trace per step: {[f"{x:.4f}" for x in trace.step_kl]}')
print(f'Top-3 predicted candidate ids:', sample.candidate_ids[trace.final_probs.topk(3).indices].tolist())
print(f'Gold entities:', sample.gold_entities.tolist())

## 10. Next steps

- **Real data**: implement `WebQSPDataset.__getitem__` in `data.py` against your `gnn_features.pkl` schema.
- **Real teacher**: replace `TeacherLLM.score_subgraph` with calls to a frozen Llama / GPT model. Cache outputs aggressively.
- **Cross-KG transfer**: train on Freebase, evaluate on Wikidata without retraining. The encoder is inductive (relation-typed only) so should transfer if relation embeddings overlap.
- **Online adaptation**: after deployment, accumulate (LAS prediction, actual LLM output) pairs and run small online distill steps to keep LAS aligned with LLM behavior drift.
